# Figure 1: Imperfect inversion

The cell below is tagged `#| label: fig-inversion-output`, so `02-gold-standard.md` embeds its output with `:::{figure} #fig-inversion-output`.

In [ ]:
#| label: fig-inversion-output
# Figure 1: effect of an imperfect inversion pulse on inversion-recovery T1 fitting.
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

# --- Parameters -------------------------------------------------------------
TI = np.array([50, 400, 1100, 2500.0])    # inversion times [ms], Barral et al. 2018
TI_fine = np.linspace(0, 3000, 300)       # for smooth curves
T1 = 1130.0                               # muscle at 1.5T, Barral et al. 2018
ANGLES = np.arange(140, 181, 1)           # actual inversion angles [deg] for the slider
DEFAULT = 155                             # typical effective angle of an adiabatic pulse

# --- Signal models -----------------------------------------------------------
def model_3p(ti, c, k, T1):
    """3-parameter IR model: k = 1 - cos(theta)."""
    return c * (1 - k * np.exp(-ti / T1))

def model_2p(ti, c, T1):
    """2-parameter IR model: assumes a perfect 180 deg inversion (k = 2)."""
    return c * (1 - 2 * np.exp(-ti / T1))

def fit_both(theta_deg):
    """Simulate noise-free data, fit both models, return data and fitted parameters."""
    k_true = 1 - np.cos(np.deg2rad(theta_deg))
    data = model_3p(TI, 1.0, k_true, T1)
    p2, _ = curve_fit(model_2p, TI, data, p0=[1, 1000], bounds=([0, 10], [10, 5000]))
    p3, _ = curve_fit(model_3p, TI, data, p0=[1, 2, 1000], bounds=([0, 0, 10], [10, 3, 5000]))
    return data, p2, p3

results = {a: fit_both(a) for a in ANGLES}
errors = np.array([100 * (results[a][1][1] - T1) / T1 for a in ANGLES])

# --- Build figure -------------------------------------------------------------
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)

# Static traces (always visible)
fig.add_trace(go.Scatter(x=[0, 3000], y=[-1, -1], mode="lines",
                         line=dict(color="gray", dash="dash", width=1),
                         name="start after a perfect 180° inversion (−1)"), row=1, col=1)
fig.add_trace(go.Scatter(x=ANGLES, y=errors, mode="lines", line=dict(color="crimson"),
                         showlegend=False), row=1, col=2)
fig.add_annotation(x=DEFAULT, y=float(errors[ANGLES == DEFAULT][0]), xref="x2", yref="y2",
                   text="typical adiabatic pulse", showarrow=True, arrowhead=2, ax=50, ay=-40)
n_static = 2

# Per-angle traces: only one set visible at a time
TRACES_PER_ANGLE = 6
for a in ANGLES:
    data, p2, p3 = results[a]
    start = np.cos(np.deg2rad(a))              # Mz right after the inversion (M0 = 1)
    err = 100 * (p2[1] - T1) / T1
    vis = bool(a == DEFAULT)
    fig.add_trace(go.Scatter(x=TI_fine, y=model_3p(TI_fine, 1, 1 - start, T1), mode="lines",
                             line=dict(color="lightgray", width=6),
                             name=f"true curve (T1 = {T1:.0f} ms)", visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, 0], y=[-1, start], mode="lines+markers",
                             line=dict(color="darkorange", width=4), marker=dict(size=6),
                             name=f"real start: {start:.2f} instead of −1", visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=TI, y=data, mode="markers", marker=dict(size=11, color="black"),
                             name="measured points", visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=TI_fine, y=model_2p(TI_fine, *p2), mode="lines",
                             line=dict(color="crimson", dash="dash"),
                             name=f"2-parameter fit: T1 = {p2[1]:.0f} ms ({err:+.1f}%)",
                             visible=vis), row=1, col=1)
    # 3-parameter fit lies exactly on the true curve, so it is only shown in the legend
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="royalblue"),
                             name=f"3-parameter fit: T1 = {p3[2]:.0f} ms (correct, on the gray curve)",
                             visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=[a], y=[err], mode="markers",
                             marker=dict(size=14, color="crimson", symbol="diamond"),
                             showlegend=False, visible=vis), row=1, col=2)

# --- Slider ---------------------------------------------------------------------
steps = []
for i, a in enumerate(ANGLES):
    visible = [True] * n_static + [False] * (TRACES_PER_ANGLE * len(ANGLES))
    start = n_static + i * TRACES_PER_ANGLE
    visible[start:start + TRACES_PER_ANGLE] = [True] * TRACES_PER_ANGLE
    steps.append(dict(method="restyle", args=[{"visible": visible}], label=f"{a}°"))

fig.update_layout(
    sliders=[dict(active=int(np.where(ANGLES == DEFAULT)[0][0]), steps=steps,
                  currentvalue=dict(prefix="Actual inversion angle: "), pad=dict(t=50))],
    height=680, margin=dict(t=30, b=230),
    # legend at the very bottom of the figure, below the slider, with space in between
    legend=dict(orientation="h", yref="container", yanchor="bottom", y=0.01, x=0),
    template="plotly_white",
)
fig.update_xaxes(title_text="TI [ms]", row=1, col=1)
fig.update_yaxes(title_text="Signal / c", range=[-1.1, 1.1], row=1, col=1)
fig.update_xaxes(title_text="Actual inversion angle [°]", row=1, col=2)
fig.update_yaxes(title_text="T1 error [%]", row=1, col=2)
fig
